# 04 - Spatiotemporal Signature Tables

Fourth notebook for the Bus Matching model: Section 1.2/1.3 of the plan
-- the in-memory GTFS shape cache (`app/gtfs_cache.py`) plus the two
coarse signature tables that make blocking (next notebook) a cheap
integer hash join instead of a per-pair spatial query.

**Grid choice**: a 300m rounded grid directly on the metric CRS
(SRID 31984) already used by `shape_geom_metric`, not `h3`/`geohash` --
no new dependency, and it's the same projection the shape cache already
uses, so no extra reprojection at query time.

**Time bucket**: 15 minutes, as `floor(epoch_seconds / 900)`. This is a
*global* bucket index (not reset per day), which is fine because `date`
is carried as its own join column throughout -- it just means bucket
ids aren't meaningful on their own, only in combination with `date`.


In [1]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg

In [2]:
_root = Path.cwd()
while not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)
os.environ.setdefault("RAW_DATA_ROOT", str(_root))
sys.path.insert(0, str(_root / "ml" / "bus_matching_model" / "app"))

In [3]:
from gtfs_cache import build_shape_cache

from opa_database.config import settings

CELL_METERS = 300
TIME_BUCKET_SECONDS = 900
SAMPLE_STEP_METERS = 250

conn = psycopg.connect(settings.db_dsn)
conn.execute("CREATE SCHEMA IF NOT EXISTS ml;")
conn.commit()
print("ml schema ready")

ml schema ready


## Stage 1 - `ml.bus_matching_device_signatures`

`SELECT DISTINCT device_id, date, cell_x, cell_y, time_bucket` over
`ml.bus_matching_avl_positions`. Can't `TABLESAMPLE` a view, so timing
was confirmed live against the underlying partition instead: a 0.5%
`TABLESAMPLE` of `silver.avl_pings_y2023m11` (~131M estimated rows)
returned 654K rows with the same `ST_Transform`/grid expressions in
under 1s, so the full ~140M-row window was expected to land in the
low-minutes range for a one-time build -- confirmed live below.


In [4]:
start = time.monotonic()
conn.execute("DROP TABLE IF EXISTS ml.bus_matching_device_signatures;")
conn.execute(
    """
    CREATE TABLE ml.bus_matching_device_signatures AS
    SELECT DISTINCT
        device_id,
        metric_timestamp::date AS date,
        floor(ST_X(ST_Transform(geom, 31984)) / %(cell)s)::int AS cell_x,
        floor(ST_Y(ST_Transform(geom, 31984)) / %(cell)s)::int AS cell_y,
        floor(extract(epoch FROM metric_timestamp) / %(bucket)s)::int AS time_bucket
    FROM ml.bus_matching_avl_positions;
    """,
    {"cell": CELL_METERS, "bucket": TIME_BUCKET_SECONDS},
)
conn.commit()
print(f"device signatures built in {time.monotonic() - start:.1f}s")

with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM ml.bus_matching_device_signatures;")
    print("rows:", cur.fetchone())

device signatures built in 129.6s


rows: (28895366,)


In [5]:
start = time.monotonic()
conn.execute(
    "CREATE INDEX bus_matching_device_signatures_join_idx "
    "ON ml.bus_matching_device_signatures (date, cell_x, cell_y, time_bucket);"
)
conn.execute("CREATE INDEX ON ml.bus_matching_device_signatures (device_id);")
conn.execute("ANALYZE ml.bus_matching_device_signatures;")
conn.commit()
print(f"device signature indexes built in {time.monotonic() - start:.1f}s")

device signature indexes built in 13.9s


## Stage 2 - `ml.bus_matching_bus_signatures`

Not a SQL query: per the plan, walk each valid trip's GTFS shape(s) --
both directions, since direction is unknown at this stage -- and emit
the cells it passes through, timestamps assigned proportionally to
distance along the route. This has to happen in Python against the
`gtfs_cache` numpy arrays, not PostGIS, per the plan's core speed
argument.

**Sampling**: a fixed spatial step (250m, slightly under the 300m cell
size so no cell along the path gets skipped) per shape, computed *once
per shape* (1,864 of them) rather than per trip -- the sample
`(fraction_along_route, cell_x, cell_y)` triple only depends on the
shape, not on any individual trip's timings. Per trip, only the
timestamp mapping (`trip_start + fraction * duration`) differs, so
trips are grouped by `(feed_version_date, shape_id)` and that mapping
is applied as one vectorized numpy broadcast per group instead of a
Python loop per trip.

**Why no cross-date dedup pass is needed**: `date` is part of the
signature tuple, so two different dates can never collide -- deduping
within each date (via `pandas.drop_duplicates`) is already the full
global dedup, confirmed live below (2023-11-01 alone: 27,569 trips ->
2.68M raw samples -> 1.89M deduped rows, in 0.6s total). That lets each
date be processed and appended independently, keeping memory bounded
instead of holding the whole month's raw samples at once.

Trips whose `(gtfs_feed_version_date, gtfs_shape_id_i/v)` isn't a
`gtfs_cache` key are silently skipped for that direction -- this is the
plan's "missing data is never negative evidence" rule: a trip missing
its GTFS shape just doesn't contribute a bus signature, it isn't scored
against anything.


In [6]:
cache = build_shape_cache(conn)

sample_grid: dict[tuple, tuple[np.ndarray, np.ndarray, np.ndarray]] = {}
for key, shape in cache.items():
    n = max(int(np.ceil(shape.total_length / SAMPLE_STEP_METERS)) + 1, 5)
    fractions = np.linspace(0.0, 1.0, n)
    chainage = fractions * shape.total_length
    idx = np.searchsorted(shape.seg_cum_start, chainage, side="right") - 1
    idx = np.clip(idx, 0, len(shape.seg_len) - 1)
    seg_frac = np.clip(
        (chainage - shape.seg_cum_start[idx]) / np.maximum(shape.seg_len[idx], 1e-9),
        0,
        1,
    )
    xy = shape.seg_start[idx] + seg_frac[:, None] * shape.seg_vec[idx]
    cell_x = np.floor(xy[:, 0] / CELL_METERS).astype(np.int64)
    cell_y = np.floor(xy[:, 1] / CELL_METERS).astype(np.int64)
    sample_grid[key] = (fractions, cell_x, cell_y)

print(f"sample grids built for {len(sample_grid)} shapes")

sample grids built for 1864 shapes


In [ ]:
def bus_signature_rows_for_date(
    conn: psycopg.Connection, trip_date: str
) -> pd.DataFrame:
    """Compute deduped bus-signature rows for one trip_date."""
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT bus_id, trip_date,
                   extract(epoch FROM trip_start_timestamp)::float8 AS start_epoch,
                   extract(epoch FROM trip_end_timestamp)::float8 AS end_epoch,
                   gtfs_feed_version_date, gtfs_shape_id_i, gtfs_shape_id_v
            FROM ml.trip_validity_final
            WHERE is_valid AND trip_date = %(trip_date)s;
            """,
            {"trip_date": trip_date},
        )
        cols = [d.name for d in cur.description]
        trips = pd.DataFrame.from_records(cur.fetchall(), columns=cols)

    parts = []
    for direction_col in ("gtfs_shape_id_i", "gtfs_shape_id_v"):
        sub = trips.dropna(subset=["gtfs_feed_version_date", direction_col])
        for (feed, shape_id), g in sub.groupby(
            ["gtfs_feed_version_date", direction_col]
        ):
            key = (feed, shape_id)
            if key not in sample_grid:
                continue
            fractions, cell_x, cell_y = sample_grid[key]
            n = len(fractions)
            k = len(g)
            start_epoch = g["start_epoch"].to_numpy()[:, None]
            end_epoch = g["end_epoch"].to_numpy()[:, None]
            ts = start_epoch + fractions[None, :] * (end_epoch - start_epoch)
            time_bucket = np.floor(ts / TIME_BUCKET_SECONDS).astype(np.int64)
            parts.append(
                pd.DataFrame(
                    {
                        "bus_id": np.repeat(g["bus_id"].to_numpy(), n),
                        "date": np.repeat(g["trip_date"].to_numpy(), n),
                        "cell_x": np.tile(cell_x, k),
                        "cell_y": np.tile(cell_y, k),
                        "time_bucket": time_bucket.ravel(),
                    }
                )
            )
    if not parts:
        return pd.DataFrame(
            columns=["bus_id", "date", "cell_x", "cell_y", "time_bucket"]
        )
    return pd.concat(parts, ignore_index=True).drop_duplicates()

In [8]:
conn.execute("DROP TABLE IF EXISTS ml.bus_matching_bus_signatures;")
conn.execute(
    """
    CREATE TABLE ml.bus_matching_bus_signatures (
        bus_id      text NOT NULL,
        date        date NOT NULL,
        cell_x      integer NOT NULL,
        cell_y      integer NOT NULL,
        time_bucket bigint NOT NULL
    );
    """
)
conn.commit()

with conn.cursor() as cur:
    cur.execute(
        "SELECT DISTINCT trip_date FROM ml.trip_validity_final "
        "WHERE is_valid ORDER BY trip_date;"
    )
    trip_dates = [row[0] for row in cur.fetchall()]
print(f"{len(trip_dates)} trip dates to process:", trip_dates[0], "..", trip_dates[-1])

30 trip dates to process: 2023-11-01 .. 2023-11-30


In [ ]:
start = time.monotonic()
total_rows = 0
for trip_date in trip_dates:
    day_df = bus_signature_rows_for_date(conn, trip_date)
    if day_df.empty:
        continue
    with (
        conn.cursor() as cur,
        cur.copy(
            "COPY ml.bus_matching_bus_signatures "
            "(bus_id, date, cell_x, cell_y, time_bucket) FROM STDIN"
        ) as copy,
    ):
        for row in day_df.itertuples(index=False):
            copy.write_row(row)
    conn.commit()
    total_rows += len(day_df)

elapsed = time.monotonic() - start
print(
    f"loaded {total_rows} bus-signature rows across {len(trip_dates)} dates "
    f"in {elapsed:.1f}s"
)

In [10]:
start = time.monotonic()
conn.execute(
    "CREATE INDEX bus_matching_bus_signatures_join_idx "
    "ON ml.bus_matching_bus_signatures (date, cell_x, cell_y, time_bucket);"
)
conn.execute("CREATE INDEX ON ml.bus_matching_bus_signatures (bus_id);")
conn.execute("ANALYZE ml.bus_matching_bus_signatures;")
conn.commit()
print(f"bus signature indexes built in {time.monotonic() - start:.1f}s")

bus signature indexes built in 30.3s


## Sanity checks

In [ ]:
with conn.cursor() as cur:
    cur.execute(
        "SELECT count(*), count(DISTINCT bus_id) FROM ml.bus_matching_bus_signatures;"
    )
    print("bus_signatures rows / distinct bus_id:", cur.fetchone())

    cur.execute(
        "SELECT count(*), count(DISTINCT device_id) "
        "FROM ml.bus_matching_device_signatures;"
    )
    print("device_signatures rows / distinct device_id:", cur.fetchone())

    cur.execute(
        """
        SELECT count(*) FROM ml.trip_validity_final f
        WHERE f.is_valid AND NOT EXISTS (
            SELECT 1 FROM ml.bus_matching_bus_signatures s
            WHERE s.bus_id = f.bus_id AND s.date = f.trip_date
        );
        """
    )
    print("valid trips whose bus-date got zero signature rows:", cur.fetchone())

    cur.execute(
        """
        SELECT count(DISTINCT bus_id || trip_date::text) FROM ml.trip_validity_final f
        WHERE f.is_valid AND NOT EXISTS (
            SELECT 1 FROM ml.bus_matching_bus_signatures s
            WHERE s.bus_id = f.bus_id AND s.date = f.trip_date
        );
        """
    )
    print("distinct bus-dates with zero signature rows:", cur.fetchone())